## Import

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import torch
import matplotlib.pyplot as plt

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

%load_ext autoreload
%autoreload 2

## Dataset

In [ ]:
from datasets.Student_t import MultivariateStudentT

n, d = 10000, 16                                 # num of samples; data dimensionality
df = 3                                           # degrees of freedom
rho = 0.7

# Build block-diagonal dispersion matrix with pairwise correlation rho
dispersion = np.eye(d)
for i in range(d // 2):
    dispersion[i, i + d // 2] = rho
    dispersion[i + d // 2, i] = rho

dataset = MultivariateStudentT(dim_x=d // 2, dim_y=d // 2, df=df, dispersion=dispersion)

X, Y = dataset.sample(n_points=n)
X = torch.Tensor(X).to(device)
Y = torch.Tensor(Y).to(device)
Z = torch.cat([X, Y], dim=1)

H = dataset.entropy()
MI = dataset.mutual_information()

print('data size', Z.size())
print('entropy', H)
print('true MI', MI)

## Copula estimate

In [ ]:
from GC import GC

gc = GC()
gc.learn(Z)
log_probs = gc.log_probs(Z)

H_gc = -log_probs.mean().item()

print('H', H)
print('H_gc', H_gc)

## Visualizing marginals

In [ ]:
from scipy.stats import gaussian_kde

fig, axes = plt.subplots(1, 2, figsize=(15, 4))

for j in range(2):
    ax = axes[j]
    J = j * d // 2

    x = Z[:, J].cpu().numpy()
    kde = gc.marginals[J]
    x_grid = np.linspace(np.min(x) - 1, np.max(x) + 1, 1000)
    kde_vals = kde.evaluate(x_grid)

    ax.hist(x, bins=30, density=True, alpha=0.5, label='Histogram')
    ax.plot(x_grid, kde_vals, label='KDE', color='black')
    ax.set_title(f'Dimension {j + 1}')
    ax.legend()

plt.tight_layout()
plt.show()